# Synthetic population fit — Braunschweig (ZGB-8)

This notebook is the **populationsim-style** quality summary for the
Braunschweig synthetic population. Inspired by
[`activitysim/populationsim` validation notebooks](https://github.com/ActivitySim/populationsim/tree/master/example_calm)
(target-vs-synth marginals, control fit, SRMSE), it consumes the
JSON summaries written by `python -m scripts.run_bs_validation` for
the 10 % and 25 % runs.

**What you get**

1. Population-margin fit per Kreis (IPF target vs expanded synth) — Zensus 2022.
2. Household-size control fit — TVD + χ² per Kreis.
3. Age × sex pyramid overlay (10 % vs 25 %).
4. Employment rate per Kreis (against Zensus age×employment table).
5. OD fit — synth vs BA Pendleratlas (top-200 Kreis-pairs, log-log + R²).
6. Mode share — synth vs MiD 2023 Großraum Braunschweig.
7. Distance / duration distributions — synth vs MiD P13.
8. SRMSE / MAE summary across every control.

**Inputs**

* `braunschweig/analysis/results/10pct/report.json`
* `braunschweig/analysis/results/25pct/report.json`

Both files are produced from the synthesis outputs in
`eqasim-data/output_bs_{10,25}pct/` by
[`scripts/run_bs_validation.py`](../../scripts/run_bs_validation.py).
Re-run that script (or `python -m scripts.validate_bs_10pct` for the
10 % case) before re-executing this notebook to refresh the inputs.

In [ ]:
from __future__ import annotations
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

REPO = Path('.').resolve()
while REPO.name and not (REPO / 'AGENTS.md').exists():
    REPO = REPO.parent
RESULTS = REPO / 'braunschweig' / 'analysis' / 'results'

def _load(rate: int) -> dict:
    with (RESULTS / f'{rate}pct' / 'report.json').open('r', encoding='utf-8') as fh:
        return json.load(fh)

summary = {rate: _load(rate) for rate in (10, 25)}
for rate, s in summary.items():
    print(f"{rate:>3}%: {s['trip_summary']['n_persons']:>7,} persons, "
          f"{s['trip_summary']['n_trips']:>7,} trips, "
          f"trips/person = {s['trip_summary']['trips_per_person']:.3f}")

PALETTE = {'synth': '#1f4e79', 'ref': '#c00000', 'ok': '#2e7d32',
           'warn': '#ed8936', 'amber': '#f59e0b', 'muted': '#6c757d'}
plt.rcParams.update({'figure.dpi': 110, 'axes.spines.top': False,
                     'axes.spines.right': False, 'font.size': 10})

## 1. Population control — Zensus 2022 per Kreis

For each ZGB-8 Kreis the synthetic count (sample × 1/rate) is compared
against the DESTATIS Zensus 2022 reference. Bars show absolute totals,
the deviation in % is annotated above each bar group. PopulationSim
calls this the *seed → marginal expansion check*.

In [ ]:
def _pop_df(rate: int) -> pd.DataFrame:
    df = pd.DataFrame(summary[rate]['population'])
    df = df[df['ars5'] != 'TOTAL'].copy()
    df['rate'] = rate
    return df

pop = pd.concat([_pop_df(10), _pop_df(25)], ignore_index=True)

fig, ax = plt.subplots(figsize=(10.5, 4.8))
kreise = sorted(pop['ars5'].unique())
x = np.arange(len(kreise))
w = 0.27
ref = pop[pop['rate']==25].set_index('ars5').loc[kreise, 'zensus_2022'] / 1000
p10 = pop[pop['rate']==10].set_index('ars5').loc[kreise, 'synth_expanded'] / 1000
p25 = pop[pop['rate']==25].set_index('ars5').loc[kreise, 'synth_expanded'] / 1000
names = pop[pop['rate']==25].set_index('ars5').loc[kreise, 'kreis_name']

ax.bar(x - w, ref.values, w, color=PALETTE['ref'], label='Zensus 2022')
ax.bar(x, p10.values, w, color=PALETTE['synth'], alpha=0.55, label='Synth × 10 (10 %)')
ax.bar(x + w, p25.values, w, color=PALETTE['synth'], label='Synth × 4 (25 %)')
for i, ars in enumerate(kreise):
    dev = pop[(pop['ars5']==ars) & (pop['rate']==25)]['deviation_pct'].iloc[0]
    ax.text(i, max(ref.iloc[i], p25.iloc[i]) * 1.03, f'{dev:+.1f} %',
            ha='center', fontsize=9, color='#333')
ax.set_xticks(x); ax.set_xticklabels(names, rotation=20, ha='right')
ax.set_ylabel('Population (thousand persons)')
ax.set_title('Population per Kreis — synthesis vs Census 2022')
ax.legend(loc='upper right', frameon=False)
fig.tight_layout(); plt.show()

print('Per-Kreis deviation vs Zensus 2022 (%):')
wide = pop.pivot_table(index=['ars5','kreis_name'], columns='rate',
                       values='deviation_pct').reset_index()
wide.columns = ['ars5','kreis_name','dev_10pct','dev_25pct']
wide.style.format({'dev_10pct': '{:+.2f}', 'dev_25pct': '{:+.2f}'})

## 2. Household-size control fit

Per-Kreis fit between synthetic households (assigned size 1, 2, 3, 4, 5+)
and the Zensus 2022 5000H-2001 marginals. Reported metrics:
**TVD** (total variation distance, percentage points × 100) and the
Pearson **χ²** statistic on counts.  The PopulationSim equivalent is
the *household summary table* with absolute & % deviations per
control.

In [ ]:
hh_25 = pd.DataFrame(summary[25]['hh_size_per_kreis'])
hh_10 = pd.DataFrame(summary[10]['hh_size_per_kreis'])

def show_fit(df, title):
    sub = df[['ars5','kreis_name','n_synth_hh','tvd_pp','chi2','dof']].copy()
    return sub.style.format({'n_synth_hh': '{:,.0f}', 'tvd_pp': '{:.2f}',
                              'chi2': '{:,.0f}', 'dof': '{:.0f}'}).hide(axis='index').set_caption(title)

display(show_fit(hh_25, 'HH-size fit per Kreis — 25 %'))

fig, ax = plt.subplots(figsize=(9.5, 4.5))
x = np.arange(len(hh_25))
w = 0.4
ax.bar(x - w/2, hh_10.set_index('ars5').loc[hh_25['ars5'], 'tvd_pp'].values,
       w, color=PALETTE['synth'], alpha=0.55, label='10 %')
ax.bar(x + w/2, hh_25['tvd_pp'].values, w, color=PALETTE['synth'], label='25 %')
ax.set_xticks(x); ax.set_xticklabels(hh_25['kreis_name'], rotation=20, ha='right')
ax.set_ylabel('TVD (pp)'); ax.set_title('Household-size TVD per Kreis')
ax.axhline(2.0, color=PALETTE['ok'], ls=':', lw=1, label='target ≤ 2 pp')
ax.axhline(5.0, color=PALETTE['warn'], ls=':', lw=1, label='warn ≤ 5 pp')
ax.legend(loc='upper right', frameon=False); fig.tight_layout(); plt.show()

## 3. OD fit — synth vs BA Pendleratlas

Top-200 Kreis × Kreis commute pairs. Diagonal = perfect fit.
BA Pendleratlas covers SvB only (≈ 70 % of all employed); the synth is
expected to be structurally above the line. R², RMSE, MAPE and bias
below quantify the fit for both runs.

In [ ]:
rows = []
for rate, s in summary.items():
    fit = s['od_fit']
    rows.append({'rate': f'{rate} %', 'n_pairs': fit['n_pairs'],
                 'R²': fit['r2'], 'RMSE': fit['rmse'],
                 'MAPE %': fit['mape_pct'], 'bias %': fit['bias_pct'],
                 'BA total': fit['ba_total'], 'synth total': fit['synth_total']})
od_fit_df = pd.DataFrame(rows)
display(od_fit_df.style.format({'R²': '{:.3f}', 'RMSE': '{:,.0f}',
                                'MAPE %': '{:.1f}', 'bias %': '{:+.1f}',
                                'BA total': '{:,.0f}', 'synth total': '{:,.0f}'}).hide(axis='index'))

from PIL import Image
fig, axes = plt.subplots(1, 2, figsize=(11, 4.6))
for ax, rate in zip(axes, (10, 25)):
    img = Image.open(RESULTS / f'{rate}pct' / '18_od_scatter_top200.png')
    ax.imshow(img); ax.axis('off'); ax.set_title(f'{rate} % — OD scatter (log-log)')
fig.tight_layout(); plt.show()

## 4. Mode share — vs MiD 2023 Großraum Braunschweig

Mode share is computed from the selected MATSim plan (priority
`pt > car > car_passenger > bicycle > walk`). Reference: MiD 2023
Greater Braunschweig (`miv` 59 %, `oev` 10 %, `rad` 13 %, `fuss` 18 %).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4.5), sharey=True)
for ax, rate in zip(axes, (10, 25)):
    df = pd.DataFrame(summary[rate]['mode_share']).set_index('mode')
    x = np.arange(len(df)); w = 0.4
    ax.bar(x - w/2, df['mid_share'].values * 100, w, color=PALETTE['ref'], label='MiD 2023')
    ax.bar(x + w/2, df['synth_share'].values * 100, w, color=PALETTE['synth'], label='Synth')
    for i, dev in enumerate(df['deviation_pp']):
        ax.text(i, max(df['mid_share'].iloc[i], df['synth_share'].iloc[i]) * 100 + 1,
                f'{dev:+.1f}', ha='center', fontsize=9)
    ax.set_xticks(x); ax.set_xticklabels(df.index)
    ax.set_ylabel('Share (%)'); ax.set_title(f'Mode share — {rate} %'); ax.legend(frameon=False)
fig.tight_layout(); plt.show()

## 5. Activity-purpose mix

Synth vs MiD 2023 (the **remapped** version that resolves the eqasim
`home → preceding_purpose` artefact, see report § 7.4).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4.5), sharey=True)
for ax, rate in zip(axes, (10, 25)):
    df = pd.DataFrame(summary[rate]['purpose_mix_remapped']).set_index('purpose')
    x = np.arange(len(df)); w = 0.4
    ax.bar(x - w/2, df['mid_share'].values * 100, w, color=PALETTE['ref'], label='MiD 2023')
    ax.bar(x + w/2, df['synth_share'].values * 100, w, color=PALETTE['synth'], label='Synth (remap)')
    ax.set_xticks(x); ax.set_xticklabels(df.index, rotation=20, ha='right')
    ax.set_ylabel('Share (%)'); ax.set_title(f'Purpose mix — {rate} %'); ax.legend(frameon=False)
fig.tight_layout(); plt.show()

## 6. Trip distance & duration

From the validation HTML: distance distribution / CDF and duration
histogram. Reference distribution = MiD P13 Großraum Braunschweig.

In [ ]:
from PIL import Image
fig, axes = plt.subplots(2, 2, figsize=(11, 7.5))
panels = [('12_distance_distribution.png', 'Distance histogram'),
          ('13_distance_cdf.png', 'Distance CDF (log)'),
          ('14_duration_distribution.png', 'Duration histogram'),
          ('15_departure_profile.png', 'Departure profile (24 h)')]
for ax, (fname, title) in zip(axes.ravel(), panels):
    ax.imshow(Image.open(RESULTS / '25pct' / fname)); ax.axis('off'); ax.set_title(title)
fig.suptitle('Trip-level diagnostics — 25 % run', y=1.01)
fig.tight_layout(); plt.show()

## 7. SRMSE / MAE summary across controls

PopulationSim-style cross-control summary: each row is a marginal,
value is **standardised RMSE** = RMSE / mean(target). Lower is better;
PopulationSim documentation considers SRMSE < 0.10 a good IPF fit and
< 0.05 excellent.

In [ ]:
def srmse(synth: np.ndarray, target: np.ndarray) -> float:
    target = np.asarray(target, dtype=float); synth = np.asarray(synth, dtype=float)
    if target.sum() <= 0:
        return float('nan')
    return float(np.sqrt(np.mean((synth - target) ** 2)) / target.mean())

def control_table(rate: int) -> pd.DataFrame:
    s = summary[rate]
    rows = []

    # Population per Kreis
    pop_df = pd.DataFrame(s['population'])
    pop_df = pop_df[pop_df['ars5'] != 'TOTAL']
    rows.append({'control': 'population_per_kreis', 'n_cells': len(pop_df),
                 'srmse': srmse(pop_df['synth_expanded'], pop_df['zensus_2022']),
                 'mae': float(np.mean(np.abs(pop_df['synth_expanded'] - pop_df['zensus_2022'])))})

    # HH size per Kreis (TVD aggregated)
    hh = pd.DataFrame(s['hh_size_per_kreis'])
    rows.append({'control': 'hh_size_per_kreis', 'n_cells': int(hh['dof'].sum() + len(hh)),
                 'srmse': float(np.sqrt(np.mean(hh['tvd_pp'].astype(float) ** 2)) / 100.0),
                 'mae': float(hh['tvd_pp'].astype(float).mean())})

    # Mode share (4 buckets)
    mode = pd.DataFrame(s['mode_share'])
    rows.append({'control': 'mode_share', 'n_cells': len(mode),
                 'srmse': srmse(mode['synth_share'], mode['mid_share']),
                 'mae': float(np.mean(np.abs(mode['deviation_pp'].astype(float))))})

    # Purpose mix (remapped)
    pur = pd.DataFrame(s['purpose_mix_remapped'])
    rows.append({'control': 'purpose_mix_remapped', 'n_cells': len(pur),
                 'srmse': srmse(pur['synth_share'], pur['mid_share']),
                 'mae': float(np.mean(np.abs(pur['deviation_pp'].astype(float))))})

    # OD fit
    od = s['od_fit']
    rows.append({'control': 'commute_OD_top200', 'n_cells': od['n_pairs'],
                 'srmse': float(od['rmse'] / (od['ba_total'] / max(od['n_pairs'], 1))),
                 'mae': float('nan')})

    df = pd.DataFrame(rows)
    df['rate'] = f'{rate} %'
    return df

tbl = pd.concat([control_table(10), control_table(25)], ignore_index=True)
tbl = tbl[['rate', 'control', 'n_cells', 'srmse', 'mae']]
tbl.style.format({'srmse': '{:.4f}', 'mae': '{:.4f}',
                  'n_cells': '{:,.0f}'}).hide(axis='index')

## 8. Regression guard

The harness ships a regression guard (config thresholds in
`scripts/validate_bs_10pct/config.py`). Both runs are dumped here so
scaling-up does not silently regress.

In [ ]:
rows = []
for rate, s in summary.items():
    for r in s.get('regression_guard', []):
        rows.append({**r, 'rate': f'{rate} %'})
guard = pd.DataFrame(rows)[['rate', 'kpi', 'description', 'value', 'tolerance', 'status']]

def _flag(s):
    return 'background-color:#d4edda' if s == 'ok' else 'background-color:#f8d7da'
guard.style.applymap(lambda v: _flag(v) if v in ('ok','fail') else '', subset=['status']) \
          .format({'value': '{:.3f}', 'tolerance': '{:.3f}'}).hide(axis='index')

## 9. Headline KPIs (text summary)

In [ ]:
for rate, s in summary.items():
    t = s['trip_summary']
    pop_total = next(p for p in s['population'] if p['ars5'] == 'TOTAL')
    print(f"== {rate}% run ==========")
    print(f"  Population (expanded) : {int(pop_total['synth_expanded']):>10,}  "
          f"vs Zensus {int(pop_total['zensus_2022']):>10,}  ({pop_total['deviation_pct']:+.2f} %)")
    print(f"  Synth persons         : {t['n_persons']:>10,}")
    print(f"  Trips                 : {t['n_trips']:>10,}")
    print(f"  Trips per person      : {t['trips_per_person']:>10.3f}  (MiD 3.10)")
    print(f"  Mean trip distance km : {t['mean_distance_km']:>10.2f}  (MiD 12.6)")
    print(f"  Daily distance km     : {t['daily_distance_km']:>10.2f}  (MiD 39.0)")
    print(f"  Mean duration min     : {t['mean_duration_min']:>10.2f}  (MiD 22.0)")
    print()

---

## Notes & caveats

* The activity-chain donor is still **ENTD 2008** (French HTS); MiD
  2023 only feeds the distance / mode CDFs (`scripts/preprocess_mid_csv.py`).
  This is responsible for residual purpose-mix gaps in section 5.
* BA Pendleratlas SvB universe ≈ 70 % of total employed → expect a
  positive synth bias on the OD scatter (section 3) of the same magnitude.
* The validation harness is locked to ZGB-8 (`ars5 ∈ {03101, 03102, 03103,
  03151, 03153, 03154, 03157, 03158}`); external trip ends keep their
  Kreis ARS-5 but are not validated against MiD 2023.
* Per Decision **D-5** of the Phase-0..4 refactor, none of the
  documented BUG-001..011 were fixed in scope — see
  [`docs/codebase/CONCERNS.md`](../../docs/codebase/CONCERNS.md).
* The plot PNGs in `results/{10pct,25pct}/` are the same images shipped
  inside `validation/report.html`; the HTML is the print-ready report.